## 1. Introduction

<div style="padding: 5px; border-left: 5px solid #ceb431; margin-bottom: 10px;">
    <strong>October 29th, 2026</strong>
</div>

In the last notebook, we implemented the init and h-gate methods for our state vector simulation, as well as a binary index value debugging method. 

In this notebook, we'll continue our development by understanding the role and implementation of the Pauli-X (NOT) and CNOT (Controlled-NOT) gates.

## 2. Understanding the Gates

My understanding of the Pauli-X gate at this point is that it is like the Hadamard gate from notebook 3, in that it is directly applied to a single cubit, except it completely flips the arrow to point to the other side of the bloch sphere instead of just changing it by 90 degrees.

Because I knew I was coming up on CNOT, a concept I've been struggling to mentally model, I went ahead and went to the library after class to try to wrap my head around it by drawing it out.

![understandingcnot.png](../assets/images/understandingcnot.jpg)

After staring at the board and really trying to focus on what makes entanglement such a unique state, I believe I understand it enough to start programming the NOT and CNOT methods. Drawing the bloch sphere representation of each qubit really helped make it clear why we physically cannot seperate the values of the 2 qubits once entangled. 

## 2. Building the Pauli-X (NOT) Gate

I'm wondering whether we really need to iterate through the indices like we need for the H-gate method. Since we're just taking half of all possibilites and swapping that value with it's pair when the target qubit is opposite, it seems like we could just find that split a lot easier and blanket apply them all.

Visualizing the binary form of some simple indices:


$|000\rangle$

$|001\rangle$

$|010\rangle$

$|011\rangle$

$|100\rangle$

$|101\rangle$

$|110\rangle$

$|111\rangle$

So, the NOT version of qubit 0 (indexed from the right, z in xyz) is the very next element in the list. The NOT version of qubit 1 is just the element after that. 

It seems like you just skip i elements where i is the index of the target qubit (indexed from right)

ex:

q2|q1|q0 

.px(q0) = skip 0 elements (next element)

.px(q1) = skip 1 element (ex: 000 -> IGNORE[001] -> 010)

.px(q2) = skip 2 elements (ex: 000 -> IGNORE[001] -> IGNORE[010] -> 011)

**...oh... nevermind**

WAIT, by looking at it and testing by looking up 8 bit representations of numbers 0-32, I believe I have found the real pattern that can be applied:

It seems like to find the "not-pair" of a binary index, it is:

not-pair array index of x = x+2^i where i is the target qubit's index (indexed from right)

ex:

starting from state_vector array index 0, binary: 000

q2|q1|q0 

.px(q0) = $0 + 2^0 = 1$, in binary: 001 PERFECT!

.px(q1) = $0 + 2^1 = 2$, in binary: 010 PERFECT!

.px(q2) = $0 + 2^2 = 4$, in binary: 100 PERFECT!

starting from state_vector array index 4, binary 100

.px(q0) = $4 + 2^0 = 5$, in binary: 101 PERFECT!

.px(q1) = $4 + 2^1 = 6$, in binary: 110 PERFECT!

.px(q2) = $4 + 2^2 = 8$, in binary: 1000 ??

OH this would be in the latter half array in a 3 qubit circuit. Even so, it calculates 8, which on a 0-7 possibility array would still be correctly 000 if it wrapped. Time to try an odd, non-zero larger example just to make sure: 

In a 9 qubit circuit with 512 possibilites:

starting from 000000000:

q8|q7|q6|q5|q4|q3|q2|q1|q0 

.px(q4) = $0 + 2^4 = 16$, in binary: 000010000 PERFECT!

.px(q7) = $0 + 2^7 = 128$, in binary: 010000000 PERFECT!

.px(q8) = $0 + 2^4 = 256$, in binary: 100000000 PERFECT!

starting from 100000000:

.px(q5) = $256 + 2^5 = 288$, in binary: 100100000 PERFECT!

Seems promising! Makes sense that it's 2^n related given how common 2^n is in this stuff.

I would normally do .copy() to not have just created a reference to half the list. However, making that copy is an O(n) operation, and I'm not convinced we need to do that to make this work, possibly faster than with it.

Since all state_vector arrays will be an even number $2^n$, I'm pretty sure all we need to do conceptually is something like:

In [507]:
import shutil

shutil.copy("quantum_circuit_nb03.py", "quantum_circuit_nb04.py")

'quantum_circuit_nb04.py'

In [508]:
#%%writefile -a quantum_circuit_nb04.py 
# commented out because error found

def px(self, target_idx):
    sv = self.state_vector

    idx1 = np.array(range(self.size//2))
    idx2 = idx1 + 2**target_idx

    sv[idx1], sv[idx2] = sv[idx2], sv[idx1]


    get_bin_string = np.vectorize(np.binary_repr)
    return self, get_bin_string(idx1), get_bin_string(idx2)
    

Let's test it with a 3 qubit circuit:

```
LEGACY TESTING CODE TO SHOW ERROR (Tuple return was just for debugging):
%run -i quantum_circuit_nb04.py

three_q_circuit = QuantumCircuit(3)

three_q_circuit.show_amp_bin()

print()

three_q_circuit.h(1).show_amp_bin()

print()

x, y, z = three_q_circuit.px(1)

x.show_amp_bin()
print()
print(y)
print()
print(z)
```

```
LEGACY OUTPUT:
1.0: 000
0.0: 001
0.0: 010
0.0: 011
0.0: 100
0.0: 101
0.0: 110
0.0: 111

0.7071067811865475: 000
0.0: 001
0.7071067811865475: 010
0.0: 011
0.0: 100
0.0: 101
0.0: 110
0.0: 111

0.7071067811865475: 000
0.0: 001
0.7071067811865475: 010
0.0: 011
0.7071067811865475: 100
0.0: 101
0.0: 110
0.0: 111
```

Okay, that error was terribly forseeable AND hard to fix, but I think I've got it:

The reason it was erroring is because taking the first half of the array does not only leave one pair for each pair. Possibility 000 is array index 0 and 010 is array index 2. That means the half includes both pairs, and worse, that means the half must NOT have 1 from each pair. 

To fix this, I believe I need to change the indexes being put into idx1 to be something along the lines that I was getting to on the board:

![bugfixingcropped.jpg](../assets/images/bugfixingcropped.jpg)

Seems like it just needs to be idx1 * 2 when it is even and idx1 * 2 -1 when it is odd. However, I need to extend the test cases to confirm the pattern:

[0, 1, 2, 3, 4, 5,  6,  7]
[0, 1, 4, 5, 8, 9, 12, 13]

The pattern seems to hold, and there's no reason to think that would change with a larger test case.

The pattern can be represnted in code as idx1 = idx1*2 - 1

Let's try implementing that into our method:

In [509]:
#%%writefile -a quantum_circuit_nb04.py 
# commented out because error foundwritefile -a quantum_circuit_nb04.py```

def px(self, target_idx):
    sv = self.state_vector

    idx1 = np.array(range(self.size//2))
    idx1 = (idx1 * 2) - (idx1 % 2)
    idx2 = idx1 + 2**target_idx

    sv[idx1], sv[idx2] = sv[idx2], sv[idx1]

    return self

_Failed Test Deleted for readability_

Okay, I cannot believe I didn't realize that the formula for indices to idx1 values was going to change based on what neuron, especially considering that I noted that exact phenomenon earlier in the original idea phase of how to deal with this. I think it's definitely time to pack up for now and go back to bitwise operations and masks to find the pairs, while keeping my original approach of in-place replacement for efficiency.

<div style="padding: 5px; border-left: 5px solid #ceb431; margin-bottom: 10px;">
    <strong>October 30th, 2026</strong>
</div>

I need to really understand the power of these bitwise operations. I already designed the code that will swap them I just need to be able to apply it to all of them efficiently.

Earlier, I used a left shift ```<<<``` operator to create a mask, and a ```XOR``` to locate the paired index. That seemed to work really well, but let me take another crack at understanding exactly what it's doing, because I'd like to be able to intuitivley use this in future projects:

"Shifts the bits of the number to the left and fills 0 on voids right as a result. Similar effect as of multiplying the number with some power of two." - [geeksforgeeks](https://www.geeksforgeeks.org/python/python-bitwise-operators/)

It all comes back to the power of two, huh.

I'll need to test this a bit more to really grasp it:

In [510]:
# Before, we used target_qubit as the right side of the operator to create a mask like so:

example_target_qubit = 1

output = 1 << example_target_qubit

#So what does that make?

print(example_target_qubit)

print(bin(example_target_qubit))

print(output)

print(bin(output))

1
0b1
2
0b10


WAIT, it's literally just putting a 1 exactly in the qubit target index (from right) that we need. I knew this before, but for some reason up until this point it just didn't click with me, but this makes sense.

So now we can just use that to check any binary representation of a numbers value with the other operations, ```&``` can find us one where that position is one, and ```XOR``` can locate the pair, just as we did before. Even though I spent a long time on my algorithm yesterday just to come back to the same method, I'm glad I went through it to develop and understanding of why this is really the best method. I think I was just tunnel-visioned on the prospect of doing it extremely efficiently, to the point where I ignored an extremely efficient method that I had already used for the H-gate. I'll just confirm the pattern and implementation that I am referring to before implementing it in the QuantumCircuit class:

In [511]:
example_target_qubit = 2

example_mask = 1 << example_target_qubit

# EX: We want to find out if target qubit is 1 on a given index.

for i in range(10):
    print(f"Mask: {bin(example_mask)} | Binary Idx: {bin(i)} | Something: {i & example_mask}")

Mask: 0b100 | Binary Idx: 0b0 | Something: 0
Mask: 0b100 | Binary Idx: 0b1 | Something: 0
Mask: 0b100 | Binary Idx: 0b10 | Something: 0
Mask: 0b100 | Binary Idx: 0b11 | Something: 0
Mask: 0b100 | Binary Idx: 0b100 | Something: 4
Mask: 0b100 | Binary Idx: 0b101 | Something: 4
Mask: 0b100 | Binary Idx: 0b110 | Something: 4
Mask: 0b100 | Binary Idx: 0b111 | Something: 4
Mask: 0b100 | Binary Idx: 0b1000 | Something: 0
Mask: 0b100 | Binary Idx: 0b1001 | Something: 0


So it just returns 0 when there is a 0 in that target qubit's index, right, now I remember. Okay, im ready to implement this based on how ```&``` and ```XOR``` worked in the H-gate, but python loops are slow, so I'm gonna try to figure out how to not use them here. 

I believe I have figured out that np filters will allow us to just grab all of the indexes where ```index & mask == 0```, and that will give us half of each pair in the array.

From there, we should be able to just use ```XOR``` to grab the other indexes, before using the same swap trick we had initially:

In [512]:
%%writefile -a quantum_circuit_nb04.py 

    def px(self, target_idx):
        mask = 1 << target_idx
        sv = self.state_vector

        total_indices = np.arange(len(sv))

        idx1 = total_indices[(total_indices & mask) == 0]
        idx2 = idx1 ^ mask

        sv[idx1], sv[idx2] = sv[idx2].copy(), sv[idx1].copy()

        return self

Appending to quantum_circuit_nb04.py


Instead of a bunch of individual tests, I think it'd be a good time to design a larger test suite kind of thing for all future debugging of this method:

In [513]:
%run -i quantum_circuit_nb04.py

def test_paulix(n_qubits):

    np.set_printoptions(linewidth=200)

    test_circuit = QuantumCircuit(n_qubits)
    print(f"INITIALIZED: {test_circuit.state_vector.real}")

    test_circuit.h(0)
    print(f"HADAMARD q(0): {test_circuit.state_vector.real}")

    for i in range(n_qubits):
        test_circuit.px(i)
        print(f"PAULI-X q({i}): {test_circuit.state_vector.real}")

test_paulix(1)

test_paulix(2)

test_paulix(3)

test_paulix(10)

test_paulix(15)


INITIALIZED: [1. 0.]
HADAMARD q(0): [0.70710678 0.70710678]
PAULI-X q(0): [0.70710678 0.70710678]
INITIALIZED: [1. 0. 0. 0.]
HADAMARD q(0): [0.70710678 0.70710678 0.         0.        ]
PAULI-X q(0): [0.70710678 0.70710678 0.         0.        ]
PAULI-X q(1): [0.         0.         0.70710678 0.70710678]
INITIALIZED: [1. 0. 0. 0. 0. 0. 0. 0.]
HADAMARD q(0): [0.70710678 0.70710678 0.         0.         0.         0.         0.         0.        ]
PAULI-X q(0): [0.70710678 0.70710678 0.         0.         0.         0.         0.         0.        ]
PAULI-X q(1): [0.         0.         0.70710678 0.70710678 0.         0.         0.         0.        ]
PAULI-X q(2): [0.         0.         0.         0.         0.         0.         0.70710678 0.70710678]
INITIALIZED: [1. 0. 0. ... 0. 0. 0.]
HADAMARD q(0): [0.70710678 0.70710678 0.         ... 0.         0.         0.        ]
PAULI-X q(0): [0.70710678 0.70710678 0.         ... 0.         0.         0.        ]
PAULI-X q(1): [0.         0.

Well that worked amazingly, it did take some messing around with to get absolutely perfect but nothing to markdown about! I suppose this means we're ready to take on the CNOT gate!

## 3. Building the CNOT (Controlled NOT) Gate

For the CNOT gate, I'm feeling pretty confident in my ability to implement it rather quickly. I'm thinking that we just take a subsection of the sv where it only includes indices where the control qubit index is 1, then perform the NOT gate logic on only those indices on the target qubit:

In [ ]:
%%writefile -a quantum_circuit_nb04.py 

    def cnot(self, control_idx, target_idx):
        control_mask = 1 << control_idx
        target_mask = 1 << target_idx
        sv = self.state_vector

        total_indices = np.arange(len(sv))

        idxs_when_c1 = total_indices[(total_indices & control_mask) != 0]

        idxs_when_t0 = idxs_when_c1[(idxs_when_c1 & target_mask) == 0]
        idxs_when_t1 = idxs_when_t0 ^ target_mask

        sv[idxs_when_t0], sv[idxs_when_t1] = sv[idxs_when_t1].copy(), sv[idxs_when_t0].copy()

        return self

Appending to quantum_circuit_nb04.py


In [515]:
def test_cnot(n_qubits):

    np.set_printoptions(linewidth=200)

    test_circuit_cnot = QuantumCircuit(n_qubits)
    print(f"INITIALIZED: {test_circuit_cnot.state_vector.real}")

    test_circuit_cnot.h(0)
    print(f"HADAMARD q(0): {test_circuit_cnot.state_vector.real}")

    test_circuit_cnot.cnot(0, 1)
    print(f"CNOT q(0): {test_circuit_cnot.state_vector.real}")


test_cnot(2)

INITIALIZED: [1. 0. 0. 0.]
HADAMARD q(0): [0.70710678 0.70710678 0.         0.        ]


AttributeError: 'QuantumCircuit' object has no attribute 'cnot'